Complete Training Loop (Modern PyTorch)

The essential PyTorch pattern with 2.0+ best practices

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# define model

class SimpleNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(784,256),
        nn.GELU(),   # modern activation smoother than relu
        nn.Dropout(0.1), # regularization
        nn.Linear(256,10)
        # NOTE: No softmax! CrossEntropyLoss includes it.
    )

  def forward(self, x):
    return self.layers(x)

# setup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNet().to(device)

# PyTorch 2.0+: compile for 2x speedup (optional)
# model = torch.compile(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr = 1e-3, weight_decay=0.01)

print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print(f"Device: {device}")

# training loop , the 5 steps

X = torch.randn(100,784).to(device)
y = torch.randint(0, 10, (100,)).to(device) # Fixed: changed torch.randn to torch.randint for class labels

for epoch in range(5):
  model.train()

  # step 1 - zero gradients ( set_to_none = True, faster)
  optimizer.zero_grad(set_to_none = True) # Fixed: changed zero_grads to zero_grad

  # step 2 - forward pass
  output = model(X)

  # step 3 - compute loss
  loss = criterion(output, y)

  # step 4 - backward pass
  loss.backward()

  # step 5 - update weights using gradients
  optimizer.step()

  print(f"Epoch {epoch + 1}: Loss = {loss.item():.4f}")


# ============================================================
# 4. EVALUATION (always eval mode + no_grad!)
# ============================================================

model.eval()
with torch.no_grad():
  preds = model(X).argmax(dim = 1)
  acc = (preds == y).float().mean()
  print(f"Accuracy: {acc:.1%}")

Model: 203,530 params
Device: cpu
Epoch 1: Loss = 2.3215
Epoch 2: Loss = 1.9812
Epoch 3: Loss = 1.6915
Epoch 4: Loss = 1.3948
Epoch 5: Loss = 1.1304
Accuracy: 100.0%


Autograd: How Gradients Work

Understanding automatic differentiation




In [9]:
import torch

# ============================================================
# AUTOGRAD DEMO: y = x², ∂y/∂x = 2x
# ============================================================

# Create tensor with gradient tracking
x = torch.tensor([2.0, 3.0, 4.0], requires_grad=True)
print(f"x = {x.tolist()}")

# Forward: compute y = x²
y = x ** 2
print(f"y = x² = {y.tolist()}")

# Compute loss (sum of y)
loss = y.sum()
print(f"loss = sum(y) = {loss.item()}")

# Backward: compute gradients via chain rule
loss.backward()

# Access gradients: ∂loss/∂x = 2x
print(f"\nx.grad = 2x = {x.grad.tolist()}")
# Expected: [4.0, 6.0, 8.0] = 2 × [2, 3, 4]

# ============================================================
# KEY INSIGHT
# ============================================================
print("\n" + "="*40)
print("Autograd = automatic chain rule")
print("1. requires_grad=True → track operations")
print("2. .backward() → compute all gradients")
print("3. .grad → access the gradient values")

x = [2.0, 3.0, 4.0]
y = x² = [4.0, 9.0, 16.0]
loss = sum(y) = 29.0

x.grad = 2x = [4.0, 6.0, 8.0]

Autograd = automatic chain rule
1. requires_grad=True → track operations
2. .backward() → compute all gradients
3. .grad → access the gradient values
